# Outlier detection practical

In this practical, you will explore the code behind detecting anomalies in data.

## Loading a dataset


In [ ]:
from sklearn.datasets import load_boston
import pandas as pd

data = load_boston()
X = data.data
y = data.target

X = pd.DataFrame(X)
X.head()


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33


We're working with the Boston House-price dataset. It relates to predicting house prices given different features such as square footage and location. It comes built-in with sklearn.

We're just loading it into the variable `data`, and, like all built-in datasets in sklearn, we extract the features using the `.data` attribute, and extract the targets using the `.target` attribute.

The last line is just to display X in a neat tabular form using pandas. As you can see, we have 13 features with this dataset. Now let's go ahead and separate into training and test sets before we continue further.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Robust covariance matrix

Now we'll move on to using the outlier detection methods. The first is the robust covariance matrix algorithm, which is expresssed as the `EllipticEnvelope` class in sklearn.

In [ ]:
from sklearn.covariance import EllipticEnvelope

detector = EllipticEnvelope(contamination=0.03)
detector.fit(X_train)

EllipticEnvelope(assume_centered=False, contamination=0.03, random_state=None,
                 store_precision=True, support_fraction=None)

As with most sklearn estimators, training it is as simple as calling the `.fit()` method. Something worth noting is the parameter called `contamination`. It expresses how many outliers we think are already in the data. Since we put in `0.03`, this is the fraction of data that we think are outliers in the data. This ratio will determine how sensitive this detector will be.

Now it is time to check which datapoints the estimator thinks are outliers.

In [ ]:
output = detector.predict(X_train)

When the above code is run, the estimator outputs an array, which is now assigned to the variable `output`. This array will have either `1` or `-1`, where `-1` represents an outlier.
Let us print a small section of this array to peer inside.

In [ ]:
output[:5]

array([ 1,  1,  1, -1,  1])

It seems the instance in the fourth position is an outlier. Now we'll use a bit of coding to separate out the instances that are outliers.

In [ ]:
clean_data = X_train[output == 1]

We'll just select instances from X_train that have been flagged with a `1`, which stands for inliers, and assign them to the variable `clean_data`.

## One class SVM
As you know, this estimator is best for novelty detection. This means that once the estimator is fit on the training set,  it makes the assumption that the training data has no outliers, and makes predictions on new data with this assumption.

Then, when we feed it new data, it will determine wether this new instance is "novel" or in other words, it determines if the new data looks any different than the usual pattern of data.

In [ ]:
from sklearn.svm import OneClassSVM

novelty_detector = OneClassSVM(nu = 0.03, kernel='rbf', gamma=0.1)
novelty_detector.fit(X_train)

OneClassSVM(cache_size=200, coef0=0.0, degree=3, gamma=0.1, kernel='rbf',
            max_iter=-1, nu=0.03, random_state=None, shrinking=True, tol=0.001,
            verbose=False)

The above code uses the `OneClassSVM` from sklearn. The parameter `nu` has a similar effect to the parameter `contamination`. It determines the sensitivity of the model to possible outliers in the training set.

The other parameters are native to SVMs, with the kernel function being selected as a Radial Basis Function for this application.

Now it is time to predict on new data.

In [ ]:
output = novelty_detector.predict(X_test)


## Local Outlier Factor
This algorithm can be used as both an outlier and novelty detector.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
algorithm = LocalOutlierFactor(n_neighbors=10, contamination=0.05, novelty=False)

Notice that we have set the parameter `novelty` to false, so that we can use it for outlier detection. The `n_neighbors` parameter is similar to the K Nearest Neighbors algorithms, and determines the number of neighbors to assess. If this number is set larger than the number of training samples, it will use as many samples as there are available.



In [ ]:
output = algorithm.fit_predict(X_train)

This estimator is slightly different, where it uses the `fit_predict()` method. This is because it is not designed specifically for detecting outliers in unseen data. It can only use the `fit()` and `predict()` method separately if `novelty`is set to `True`.

## Isolation forest

This algorithm is also executed identically to the other estimators for outlier detection.

In [ ]:
from sklearn.ensemble import IsolationForest
algorithm = IsolationForest(behaviour='new', contamination=0.05, n_estimators=100, max_features=3)

The parameter `behaviour='new'` tells sklearn to make this estimator match the behaviour of the other anomaly detection estimators in sklearn. Apart from this, other parameters reflect that of tree algorithms.

In [ ]:
clean_data = X_train[algorithm.fit(X_train).predict(X_train)==1]
clean_data[:5]

,0,1,2,3,4,5,6,7,8,9,10,11,12
192,0.08664,45.0,3.44,0.0,0.437,7.178,26.3,6.4798,5.0,398.0,15.2,390.49,2.87
294,0.08199,0.0,13.92,0.0,0.437,6.009,42.3,5.5027,4.0,289.0,16.0,396.90,10.40
377,9.82349,0.0,18.10,0.0,0.671,6.794,98.8,1.3580,24.0,666.0,20.2,396.90,21.24
456,4.66883,0.0,18.10,0.0,0.713,5.976,87.9,2.5806,24.0,666.0,20.2,10.48,19.01
247,0.19657,22.0,5.86,0.0,0.431,6.226,79.2,8.0555,7.0,330.0,19.1,376.14,10.15


The above code executes many different methods and processes in a single line. First it uses the `algorithm` to `fit` on the training set. Then, it predicts. This results in the array of `1` and `-1` which denote outliers. Then we select the elements of X_train by compaing it with this array.

Below is an example that breaks this line up into multiple steps:

In [ ]:
algorithm.fit(X_train)
array = algorithm.predict(X_train)
clean_data = X_train[array == 1]